<a href="https://colab.research.google.com/github/appling2024/MSP/blob/Maksim/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22WV_test_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 1. Предобработка корпуса

* проводим лемматизацию и удаляем стоп-слова;
* приводем все леммы к нижнему регистру;
* добавляем чстеречные теги к словам.

Для предобработки мы будем использовать [*UDPipe*](https://ufal.mff.cuni.cz/udpipe)

In [13]:
!pip install wget

In [12]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 37.4 MB/s eta 0:00:00


In [2]:
import wget
import sys

udpipe_url = 'https://rusvectores.org/static/models/udpipe_syntagrus.model'

modelfile = wget.download(udpipe_url)
print('ok')

ok


Функция для предобработки текста

In [3]:
def process(pipeline, text='Строка', keep_pos=True, keep_punct=False):
    entities = {'PROPN'}
    named = False
    memory = []
    mem_case = None
    mem_number = None
    tagged_propn = []

    # обрабатываем текст, получаем результат в формате conllu:
    processed = pipeline.process(text)

    # пропускаем строки со служебной информацией:
    content = [l for l in processed.split('\n') if not l.startswith('#')]

    # извлекаем из обработанного текста леммы, тэги и морфологические характеристики
    tagged = [w.split('\t') for w in content if w]

    for t in tagged:
        if len(t) != 10:
            continue
        (word_id, token, lemma, pos, xpos, feats, head, deprel, deps, misc) = t
        if not lemma or not token:
            continue
        if pos in entities:
            if '|' not in feats:
                tagged_propn.append('%s_%s' % (lemma, pos))
                continue
            morph = {el.split('=')[0]: el.split('=')[1] for el in feats.split('|')}
            if 'Case' not in morph or 'Number' not in morph:
                tagged_propn.append('%s_%s' % (lemma, pos))
                continue
            if not named:
                named = True
                mem_case = morph['Case']
                mem_number = morph['Number']
            if morph['Case'] == mem_case and morph['Number'] == mem_number:
                memory.append(lemma)
                if 'SpacesAfter=\\n' in misc or 'SpacesAfter=\s\\n' in misc:
                    named = False
                    past_lemma = '::'.join(memory)
                    memory = []
                    tagged_propn.append(past_lemma + '_PROPN ')
            else:
                named = False
                past_lemma = '::'.join(memory)
                memory = []
                tagged_propn.append(past_lemma + '_PROPN ')
                tagged_propn.append('%s_%s' % (lemma, pos))
        else:
            if not named:
                if pos == 'NUM' and token.isdigit():  # Заменяем числа на xxxxx той же длины
                    continue
                tagged_propn.append('%s_%s' % (lemma, pos))
            else:
                named = False
                past_lemma = '::'.join(memory)
                memory = []
                tagged_propn.append(past_lemma + '_PROPN ')
                tagged_propn.append('%s_%s' % (lemma, pos))

    if not keep_punct:
        tagged_propn = [word for word in tagged_propn if word.split('_')[1] != 'PUNCT']
    if not keep_pos:
        tagged_propn = [word.split('_')[0] for word in tagged_propn]
    return tagged_propn

print('ok')


ok


<>:38: SyntaxWarning: invalid escape sequence '\s'
<>:38: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipython-input-1636019365.py:38: SyntaxWarning: invalid escape sequence '\s'
  if 'SpacesAfter=\\n' in misc or 'SpacesAfter=\s\\n' in misc:


In [4]:
pip install ufal.udpipe

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 938.5/938.5 kB 23.9 MB/s eta 0:00:00


In [5]:
from ufal.udpipe import Model, Pipeline
import os
import re

def tag_ud(text='Текст нужно передать функции в виде строки!', modelfile='udpipe_syntagrus.model'):
    cnt = 0
    udpipe_model_url = 'https://rusvectores.org/static/models/udpipe_syntagrus.model'
    udpipe_filename = udpipe_model_url.split('/')[-1]

    if not os.path.isfile(modelfile):
        print('UDPipe model not found. Downloading...', file=sys.stderr)
        wget.download(udpipe_model_url)

    print('\nLoading the model...', file=sys.stderr)
    model = Model.load(modelfile)
    process_pipeline = Pipeline(model, 'tokenize', Pipeline.DEFAULT, Pipeline.DEFAULT, 'conllu')

    print('Processing input...', file=sys.stderr)
    lines = text.split('\n')
    tagged = []
    for line in lines:
        # line = unify_sym(line.strip()) # здесь могла бы быть ваша функция очистки текста
        output = process(process_pipeline, text=line)
        tagged_line = ' '.join(output)
        tagged.append(tagged_line)
        cnt += 1
        if cnt%1000 == 0:
            print(cnt)
    return '\n'.join(tagged)

In [ ]:
Приступаем к обработке корпуса!

In [8]:
text = open(r'Города.txt', 'r', encoding='utf-8').read()
processed_text = tag_ud(text=text, modelfile=modelfile)
print(processed_text[:350])
with open('mytext4.txt', 'w', encoding='utf-8') as out:
    out.write(processed_text)


Loading the model...
Processing input...


Ахетатон_PROPN  город_PROPN  ФАРАОНА-ЕРЕТИКа._NOUN Эхнатон_PROPN 

в_ADP год_NOUN до_ADP наш_DET эра_NOUN на_ADP египетский_ADJ престол_NOUN вступать_VERB Аменхотеп_PROPN  iv_NUM самый_ADJ необычный_ADJ из_ADP древнеегипетский_ADJ фараон_NOUN реформа_NOUN который_PRON порождать_VERB исключительно_ADV интересный_ADJ период_NOUN в_ADP история_NOUN ег


In [14]:
import sys
import gensim, logging

logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

Одна строка - одно предложение

In [15]:
f = 'mytext4.txt'
data = gensim.models.word2vec.LineSentence(f)

2. Приступаем к обучению моделей

2.1 CBOW

Инициализируем модель. Параметры в скобочках:
* data - данные,
* size - размер вектора,
* window - размер окна наблюдения,
* min_count - мин. частотность слова в корпусе, которое мы берем,
* sg - используемый алгоритм обучение (0 - CBOW, 1 - Skip-gram))

In [16]:
model_CBOW = gensim.models.Word2Vec(data, vector_size=500, window=5, min_count=2, sg=0)

Сколько слов в модели?

In [17]:
print(len(model_CBOW.wv.key_to_index))

1501


Сохраняем модель

In [20]:
model_CBOW.save('goroda_cbow.model')

Загружаем сохраненную модель

In [22]:
from gensim.models import Word2Vec
cbow = Word2Vec.load('goroda_cbow.model')

Косинусное сходство

In [25]:
cbow.wv.similarity("город_NOUN", "столица_NOUN")

np.float32(0.99750316)

In [26]:
cbow.wv.similarity("город_NOUN", "первый_ADJ")

np.float32(0.9976671)

In [29]:
cbow.wv.similarity("первый_ADJ", "открытие_NOUN")

np.float32(0.9403762)

In [28]:
cbow.wv.similarity("наука_NOUN", "страна_NOUN")

np.float32(0.7881085)

In [30]:
for t in cbow.wv.most_similar(positive=[u'город_NOUN'], topn=30):
    print (t[0], t[1])

в_ADP 0.9994528889656067
и_CCONJ 0.9994518756866455
быть_AUX 0.999362051486969
на_ADP 0.9993432760238647
с_ADP 0.999282956123352
он_PRON 0.9992708563804626
который_PRON 0.999269425868988
что_SCONJ 0.9992603063583374
не_PART 0.9992563128471375
и_PART 0.9992292523384094
они_PRON 0.9992102384567261
из_ADP 0.9991163611412048
о_ADP 0.9991070032119751
но_CCONJ 0.9991069436073303
к_ADP 0.9990851879119873
этот_DET 0.9990623593330383
от_ADP 0.9990320801734924
свой_DET 0.9989967942237854
а_CCONJ 0.9989777207374573
по_ADP 0.9989767074584961
до_ADP 0.9989686012268066
она_PRON 0.9989378452301025
год_NOUN 0.99893718957901
время_NOUN 0.9989325404167175
стена_NOUN 0.9989158511161804
век_NOUN 0.9989053606987
как_SCONJ 0.9988997578620911
царь_NOUN 0.9988451600074768
самый_ADJ 0.9987898468971252
только_PART 0.9987815022468567


Евклидово расстояние

In [32]:
import numpy as np

euclid1 = np.linalg.norm(cbow.wv['город_NOUN'] - cbow.wv['столица_NOUN'])
euclid2 = np.linalg.norm(cbow.wv['город_NOUN'] - cbow.wv['поселение_NOUN'])
print(euclid1, euclid2)

0.51089936 0.84991336


## 2.2 Skip-Gram

In [33]:
model_sg = gensim.models.Word2Vec(data, vector_size=500, window=5, min_count=2, sg=1)

In [35]:
model_sg.save('goroda_skip-gram.model')

In [36]:
from gensim.models import Word2Vec
sg = Word2Vec.load('goroda_skip-gram.model')

In [37]:
for t in sg.wv.most_similar(positive=[u'город_NOUN'], topn=30):
    print (t[0], t[1])

здесь_ADV 0.9996927380561829
этот_DET 0.9996921420097351
Мерв_PROPN 0.9996901154518127
Пальмира_PROPN 0.9996830821037292
сам_ADJ 0.9996806979179382
Иерихон_PROPN 0.999677836894989
Дамаск_PROPN 0.9996756315231323
к_ADP 0.999675452709198
а_CCONJ 0.9996713399887085
самый_ADJ 0.9996698498725891
дамаск_PROPN 0.999668538570404
Мемфис_PROPN 0.9996680021286011
потом_ADV 0.9996662139892578
после_ADP 0.9996657371520996
другой_ADJ 0.9996646642684937
находиться_VERB 0.9996629953384399
но_CCONJ 0.9996629953384399
от_ADP 0.9996626973152161
себя_PRON 0.9996623396873474
видеть_VERB 0.9996621608734131
место_NOUN 0.9996618628501892
свой_DET 0.9996614456176758
что_SCONJ 0.9996612071990967
столица_NOUN 0.9996586441993713
уже_ADV 0.999657928943634
захватывать_VERB 0.9996574521064758
день_NOUN 0.9996569156646729
они_PRON 0.9996564388275146
делать_VERB 0.9996563792228699
мечеть_NOUN 0.9996563196182251


In [38]:
sg.wv.similarity("город_NOUN", "древний_ADJ")

np.float32(0.9996109)

In [40]:
sg.wv.similarity("город_NOUN", "великий_ADJ")

np.float32(0.9996311)

In [41]:
sg.wv.similarity("город_NOUN", "столица_NOUN")

np.float32(0.9996585)

In [43]:
sg.wv.similarity("город_NOUN", "поселение_NOUN")

np.float32(0.999464)

In [44]:
euclid3 = np.linalg.norm(sg.wv['город_NOUN'] - sg.wv['столица_NOUN'])
euclid4 = np.linalg.norm(sg.wv['город_NOUN'] - sg.wv['поселение_NOUN'])
print(euclid3, euclid4)

0.03985122 0.580368


## Коллокаты

In [46]:
import re

for t in sg.wv.most_similar(positive=[u'город_NOUN'], topn=10):
  cond = re.search(r'_(NOUN)|(ADJ)|(NUM)', t[0])
  if cond != None:
    print (t[0], t[1])

сам_ADJ 0.9996806979179382
самый_ADJ 0.9996698498725891


In [47]:
import re

for t in cbow.wv.most_similar(positive=[u'столица_NOUN'], topn=10):
  cond = re.search(r'_(NOUN)|(ADJ)|(NUM)', t[0])
  if cond != None:
    print (t[0], t[1])

город_NOUN 0.9975033402442932
